# 01 — Model, transfer and inference

Refs #19/#39. Import the existing GRU and retention implementation, inspect executed Source/Target lineage, and demonstrate CPU inference and gradients.

## Environment and artifact setup

Use Python 3.12 as documented in the repository. Install the pinned PyTorch requirement (requirements-model.txt from the CPU index, or requirements-cuda.txt from the cu126 index), plus requirements-dev.txt. Notebook presentation dependencies used for recorded validation: nbformat 5.10.4, nbclient 0.10.2, ipykernel 6.29.5, pandas 3.0.1, matplotlib 3.10.8.

Open from the repository or set HUNG_REPO_ROOT. Download/extract identified inputs and run artifacts; set HUNG_INPUT_BASE and HUNG_RUN_ROOT if locations differ. Missing files fail explicitly. Large inputs/checkpoints stay outside Git; only load trusted project checkpoints.

Default cells inspect existing evidence and run small CPU examples. They do not launch matrix training. Recorded matrix code SHA: a34a49e5f04920ef30456fe6c0ea56eb2f1d659a, diagnostic config v1 and selection v2. Clean CPU reproduction was validated in a new isolated repo-local Python 3.13.7 environment with torch 2.8.0+cpu, pip check, the full unit suite and a fresh kernel per notebook. README's reference Python 3.12 environment was not executed in this local check. The global CUDA environment has an unrelated ultralytics/numpy conflict; it was not inherited by the isolated environment.


In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path

import pandas as pd

candidates = (
    [Path(os.environ["HUNG_REPO_ROOT"])]
    if os.environ.get("HUNG_REPO_ROOT")
    else [Path.cwd(), *Path.cwd().parents]
)
REPO = next(
    (
        path.resolve()
        for path in candidates
        if (path / "experiments/hung_sprint2/model.py").is_file()
    ),
    None,
)
if REPO is None:
    raise FileNotFoundError("Set HUNG_REPO_ROOT to the checked-out repository.")
sys.path.insert(0, str(REPO))
RUN_ROOT = Path(
    os.environ.get("HUNG_RUN_ROOT", str(REPO / "runs/hung-39-representative-local-20261025T2103"))
).resolve()
INPUT_BASE = Path(
    os.environ.get("HUNG_INPUT_BASE", str(REPO / "data/inputs/issue18-input-handoff-v1-20261019"))
).resolve()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


identity = read_json(RUN_ROOT / "matrix_identity.json")
config = read_json(REPO / "experiments/hung_sprint2/development_config_v1.json")
assert identity["phase"] == "representative"
assert identity["config_sha256"] == sha256(
    REPO / "experiments/hung_sprint2/development_config_v1.json"
)
assert identity["selection_sha256"] == sha256(
    REPO / "docs/reports/hung-development-selection-20261025-v2.json"
)
print(
    {
        "run_id": RUN_ROOT.name,
        "execution_code_sha": identity["code"]["commit"],
        "device": identity["device"],
    }
)

{'run_id': 'hung-39-representative-local-20261025T2103', 'execution_code_sha': 'a34a49e5f04920ef30456fe6c0ea56eb2f1d659a', 'device': 'cuda'}


## Method and tensor contract

ForecastGRU uses a two-layer hidden-64 GRU on X_hist[B,24,7], a ReLU future-calendar MLP on C_future[B,24,5], and a Softplus head producing Y_pred[B,24]. Each target independently receives Source parameters. Source latent scales are population standard deviations fitted only on permitted Source training histories; the reference stays frozen.

\[
L_f=\frac1{24}\sum_h|\hat y_h-y_h|,\quad
L_r=\frac1{64}\sum_k\left(\frac{z_{T,k}-z_{S,k}}{s_k+\epsilon}\right)^2,\quad
L=L_f+\lambda L_r.
\]

The implementation also averages over the batch. Lambda zero means Source-initialized fine tuning, not Target-only initialization. See model.py, retention.py and training.py. Core implementation is imported, not copied into cells.

In [2]:
import torch
from experiments.hung_sprint2.retention import retention_loss
from experiments.hung_sprint2.training import forecast_mae, initialize_target, load_checkpoint

from ev_inputs.loader import iter_forecasts

torch.set_num_threads(1)
base = RUN_ROOT / "7d/seed-7"
source_manifest = read_json(base / "source/run_manifest.json")
target_manifest = read_json(base / "entity-2264/target/run_manifest.json")
source_path = base / "source/source_checkpoint.pt"
target_path = base / "entity-2264/target/target_checkpoint.pt"
assert sha256(source_path) == source_manifest["checkpoint_sha256"]
assert sha256(target_path) == target_manifest["checkpoint_sha256"]
source, _, latent_scale, _ = load_checkpoint(
    source_path, expected_binding=source_manifest["input_binding"], map_location="cpu"
)
target, _, _, _ = load_checkpoint(
    target_path, expected_binding=target_manifest["input_binding"], map_location="cpu"
)
pd.DataFrame(
    [
        {
            "stage": "Source",
            "examples": source_manifest["source_examples_used"],
            "steps": source_manifest["steps"],
            "seconds": source_manifest["duration_seconds"],
            "checkpoint": str(source_path),
        },
        {
            "stage": "Target 2264",
            "examples": target_manifest["target_examples_available"],
            "steps": target_manifest["steps"],
            "seconds": target_manifest["duration_seconds"],
            "checkpoint": str(target_path),
        },
    ]
)

,stage,examples,steps,seconds,checkpoint
0,Source,65888,2059,33.406835,D:\Đồ Án\.cache\sprint2-completion\runs\hung...
1,Target 2264,6,3,10.472155,D:\Đồ Án\.cache\sprint2-completion\runs\hung...


In [3]:
input_run = INPUT_BASE / "urbanev-SZH-7d"
forecast_input = next(
    row
    for row in iter_forecasts(
        input_run, expected_manifest_sha256=identity["input_manifest_sha256"]["7"]
    )
    if row["role"] == "development" and row["identity"][2] == "2264"
)
x = torch.tensor(forecast_input["x_hist"], dtype=torch.float32).unsqueeze(0)
calendar = torch.tensor(forecast_input["c_future"], dtype=torch.float32).unsqueeze(0)
target.eval()
with torch.no_grad():
    prediction, latent = target.forecast_and_latent(x, calendar)
assert prediction.shape == (1, 24) and latent.shape == (1, 64)
assert torch.all(prediction >= 0)
print(
    {
        "x_shape": list(x.shape),
        "calendar_shape": list(calendar.shape),
        "forecast_shape": list(prediction.shape),
        "parameter_count": sum(p.numel() for p in target.parameters()),
        "tensor_device": str(prediction.device),
    }
)
pd.DataFrame(
    {
        "horizon_hour": range(1, 25),
        "forecast_kwh": (prediction[0] * forecast_input["energy_scale"]).tolist(),
    }
)

{'x_shape': [1, 24, 7], 'calendar_shape': [1, 24, 5], 'forecast_shape': [1, 24], 'parameter_count': 41696, 'tensor_device': 'cpu'}


,horizon_hour,forecast_kwh
0,1,7.825769
1,2,7.757661
2,3,7.877618
3,4,7.543815
4,5,7.721786
5,6,7.850225
6,7,7.618264
7,8,7.243275
8,9,7.202745
9,10,7.963790


In [4]:
# One real Development pair: backward only, no optimizer step or saved update.
with (input_run / "labels.jsonl").open(encoding="utf-8") as stream:
    label = next(
        row
        for row in map(json.loads, stream)
        if row["identity"][2] == "2264" and row["origin"] == forecast_input["origin"]
    )
assert all(label["file_valid"]) and all(value is not None for value in label["energy_kwh"])
y = (
    torch.tensor(label["energy_kwh"], dtype=torch.float32).unsqueeze(0)
    / forecast_input["energy_scale"]
)
example_target, reference, _ = initialize_target(
    source, learning_rate=config["target"]["learning_rate"]
)
example_forecast, z_target = example_target.forecast_and_latent(x, calendar)
with torch.no_grad():
    z_source = reference.encode(x)
mae = forecast_mae(example_forecast, y)
retain = retention_loss(z_target, z_source, latent_scale, epsilon=config["target"]["epsilon"])
total = mae + config["target"]["retention_weight"] * retain
total.backward()
assert all(parameter.grad is None for parameter in reference.parameters())
assert all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in example_target.parameters()
)
assert torch.equal(mae + 0 * retain, mae)
print(
    {
        "forecast_loss_scaled": float(mae.detach()),
        "initial_retention": float(retain.detach()),
        "total_loss_scaled": float(total.detach()),
        "reference_frozen": True,
        "weights_updated": False,
    }
)

{'forecast_loss_scaled': 0.1292332261800766, 'initial_retention': 0.0, 'total_loss_scaled': 0.1292332261800766, 'reference_frozen': True, 'weights_updated': False}


## Interpretation

Tables come from executed manifests; the forecast is CPU inference from a saved target checkpoint. The final cell is a bounded gradient demonstration with no optimizer step. Diagnostic results do not establish effectiveness against baselines. Phase reports in docs/reports index the full lineage and remaining review gates.